In [ ]:
import torch 
from torch import nn
from torch.nn import functional as F
import random
import math
import pickle 
import inspect

# Torch compatibility: some versions do not accept `training=` in SDPA.
try:
    _sdpa_params = inspect.signature(F.scaled_dot_product_attention).parameters
except (TypeError, ValueError):
    _sdpa_params = {}
if 'training' not in _sdpa_params:
    _orig_sdpa = F.scaled_dot_product_attention
    def _sdpa_compat(q, k, v, attn_mask=None, dropout_p=0.0, is_causal=False, training=None):
        if training is False:
            dropout_p = 0.0
        return _orig_sdpa(q, k, v, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal)
    F.scaled_dot_product_attention = _sdpa_compat
device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 32
block_size = 64 
max_iters = 500
learning_rate = 1e-3
eval_iters = 50 
n_embd=128
n_head=4
n_layer=3
dropout = 0.2 
print(f"Using device: {device}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
chars=""
with open ('/content/drive/MyDrive/vocab.txt','r',encoding='utf-8') as f:
    text=f.read()
    chars=sorted(list(set(text)))
print(''.join(chars))
string_to_int={ch:i for i,ch in enumerate(chars)}
int_to_string={i:ch for i,ch in enumerate(chars)}
encode=lambda s: [string_to_int[c] for c in s]
decode=lambda l: ''.join([int_to_string[i] for i in l])
vocab_size = len(chars)
print(encode("hello world"))
print(vocab_size)

In [ ]:
chars=""
with open ('/content/drive/MyDrive/pg22566.txt', 'r', encoding="utf-8") as f:
    text = f.read()
print(len(text))
chars = sorted(list(set(text)))
vocab_size = len(chars)
string_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_string = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])
print(''.join(chars))
text = text[:1000000]
data = torch.tensor(encode(text), dtype=torch.long)
assert int(data.max()) < vocab_size, "token id exceeds vocab_size"
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]
print(f"train has {len(train_data)} tokens, val has {len(val_data)} tokens")
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data)-block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [ ]:


class SingleHeadCausalSelfAttention(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.q_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.k_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.v_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.out_proj = nn.Linear(n_embd, n_embd)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x):
        q = self.q_proj(x).unsqueeze(1)
        k = self.k_proj(x).unsqueeze(1)
        v = self.v_proj(x).unsqueeze(1)

        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=dropout if self.training else 0.0,
            is_causal=True,
        ).squeeze(1)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.k_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.v_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.out_proj = nn.Linear(n_embd, n_embd)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.size()

        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=dropout if self.training else 0.0,
            is_causal=True,
        )
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, use_multi_head=True):
        super().__init__()
        if use_multi_head and n_head > 1:
            self.impl = MultiHeadCausalSelfAttention(n_embd, n_head)
        else:
            self.impl = SingleHeadCausalSelfAttention(n_embd)

    def forward(self, x):
        return self.impl(x)

class FeedForward(nn.Module):
    def __init__(self,n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.GELU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout)
        )
    def forward(self,x):
        return self.net(x)
class Block(nn.Module):
    def __init__(self,n_embd,n_head,use_multi_head_attention=True):
        super().__init__()
        self.sa = CausalSelfAttention(n_embd, n_head, use_multi_head=use_multi_head_attention)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self,x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x
class GPTLanguageModel(nn.Module):
    def __init__(self, use_multi_head_attention=True):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks=nn.Sequential(*[Block(n_embd, n_head, use_multi_head_attention) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.token_embedding_table.weight=self.lm_head.weight
        self.apply(self._init_weights)
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    def forward(self, idx, targets=None):
        B,T=idx.size()
        token_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = token_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        if targets is None:
            loss=None
        else:
            B,T,C=logits.size()
            logits=logits.view(B*T,C)
            targets=targets.view(B*T)
            loss=F.cross_entropy(logits, targets)
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=40):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                k = min(top_k, logits.size(-1))
                v, _ = torch.topk(logits, k)
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_idx), dim=1)
        return idx
use_multi_head_attention = True  # set False to use single-head attention
model=GPTLanguageModel(use_multi_head_attention=use_multi_head_attention).to(device)
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
def get_lr(it):
    wrarmup_iters = 50 
    min_lr = 1e-4
    if it < wrarmup_iters:
        return learning_rate * it / wrarmup_iters
    decay_ratio = (it - wrarmup_iters) / (max_iters - wrarmup_iters)
    coeff=0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return max(min_lr, learning_rate * coeff)
for iter in range(max_iters):
    lr = get_lr(iter)
    optimizer.param_groups[0]['lr'] = lr
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(f'\nFinal loss: {loss.item():.4f}')


In [ ]:
prompt = "The Project Gutenberg EBook of The Picture of Dorian Gray, by Oscar Wilde\n\nThis eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with\no restrictions whatsoever. You may copy it, give it away or re-use it under the terms of the Project Gutenberg License included with this eBook or online at www.gutenberg.org. If you are not located in the United States, you will have to check the laws of the country where you are located before using this eBook."
context = torch.tensor(encode(prompt), dtype=torch.long, device=device)
generated_chars=decode(model.generate(context.unsqueeze(0), max_new_tokens=500)[0].tolist())
print(generated_chars)

In [ ]:
string_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_string = {i:ch for i,ch in enumerate(chars)}
encode=lambda s: [string_to_int[c] for c in s]
decode=lambda l: ''.join([int_to_string[i] for i in l])

In [ ]:
print(encode("hello world"))